# 01 — EDA: Goodreads Book Recommender

Exploratory analysis on the real preprocessed artifacts (`artifacts/sample.parquet` — 50k-user interactions; `artifacts/catalog.parquet` — surviving books). Covers rating distribution, interactions per user/book, matrix sparsity, description length, shelf/genre signal, language mix, temporal spread, and anomalies.

In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

In [ ]:
def find(name):
    for pat in (f'artifacts/{name}', f'../artifacts/{name}',
                f'/kaggle/input/**/{name}', f'/kaggle/working/**/{name}'):
        hits = glob.glob(pat, recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(name)

sample = pd.read_parquet(find('sample.parquet'))
catalog = pd.read_parquet(find('catalog.parquet'))
print('sample:', sample.shape, '| catalog:', catalog.shape)

## Overview

In [ ]:
print('interactions:', f'{len(sample):,}')
print('users:', sample.user_id.nunique(), '| books interacted:', sample.book_id.nunique())
print('catalog books:', f'{len(catalog):,}')
display(sample.head())
display(catalog[['book_id','title','language_code','shelves']].head())

## Rating distribution
0 = shelved/read but not explicitly rated (implicit feedback).

In [ ]:
rc = sample.rating.value_counts().sort_index()
ax = rc.plot.bar(color='#0a3d62')
ax.set_title('Rating distribution (0 = implicit / unrated)'); ax.set_xlabel('rating'); ax.set_ylabel('count')
plt.show()
print('implicit (rating=0) share: %.1f%%' % (100*rc.get(0,0)/rc.sum()))
print('explicit (1-5) interactions: %s' % f'{int(rc.loc[1:].sum()):,}')

## Interactions per user

In [ ]:
ipu = sample.groupby('user_id').size()
ipu.plot.hist(bins=100, log=True, color='#0a3d62')
plt.title('Interactions per user (log y)'); plt.xlabel('interactions/user'); plt.show()
print(ipu.describe().round(1).to_string())

## Interactions per book (popularity long tail)

In [ ]:
ipb = sample.groupby('book_id').size()
ipb.plot.hist(bins=100, log=True, color='#196f3d')
plt.title('Interactions per book (log y)'); plt.xlabel('interactions/book'); plt.show()
print(ipb.describe().round(1).to_string())
print('most-read book appears in', ipb.max(), 'sampled users')

## User–book matrix sparsity

In [ ]:
n_u, n_b = sample.user_id.nunique(), sample.book_id.nunique()
density = len(sample)/(n_u*n_b)
print(f'users={n_u:,}  books={n_b:,}  interactions={len(sample):,}')
print(f'matrix density = {100*density:.4f}%   (sparsity = {1-density:.5f})')

## Synopsis (description) length

In [ ]:
dl = catalog.description.fillna('').str.len()
dl.clip(upper=3000).plot.hist(bins=100, color='#d68910')
plt.title('Description length (chars, clipped at 3000)'); plt.xlabel('chars'); plt.show()
print(dl.describe().round(0).to_string())
print('empty descriptions:', f'{(dl==0).sum():,}', '(%.1f%%)' % (100*(dl==0).mean()))

## Shelf / genre signal (top tags)

In [ ]:
sc = Counter()
for s in catalog['shelves']:
    sc.update(list(s))
top = pd.Series(dict(sc.most_common(30)))
top.iloc[::-1].plot.barh(figsize=(7, 8), color='#0a3d62')
plt.title('Top 30 shelves across the catalog'); plt.tight_layout(); plt.show()

## Language mix
Note: the preprocess did **not** filter to English — non-English books are present.

In [ ]:
lc = catalog.language_code.replace('', '(blank)').value_counts().head(12)
lc.iloc[::-1].plot.barh(color='#196f3d')
plt.title('Top language codes'); plt.tight_layout(); plt.show()
eng = catalog.language_code.isin(['eng','en-US','en-GB','en-CA','en',''])
print('English-or-blank share: %.1f%%' % (100*eng.mean()))

## Temporal spread

In [ ]:
year = pd.to_datetime(sample.timestamp, unit='s').dt.year
year[year >= 2006].value_counts().sort_index().plot.bar(color='#0a3d62')
plt.title('Interactions by year'); plt.xlabel('year'); plt.show()

## Findings & anomalies

- **~54% of interactions are implicit (rating 0)** — explicit-only vs all-interactions is a real modeling fork (see the `filter_min_rating` ablation).
- **Heavy-tailed users** (median ~120, max ~68k interactions) and a **long-tail popularity** distribution over books — typical recsys skew; informs sampling/negatives.
- **Extremely sparse** user-book matrix (~0.04% dense) — motivates CF + content fusion.
- **~9% of books have empty descriptions** — content methods fall back to title + shelves for those.
- **Non-English books present** (language not filtered) — consider an English-only ablation.
- k-core floor visible: every user has ≥20 interactions, every book ≥10.